# CoNLL Named Entity Recognition

IMSA coursework publication copy. Saved outputs were cleared; rerun before citing results.


In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from pathlib import Path

# Prefer local cached CoNLL files to avoid re-downloading each run
default_cache = Path.home() / ".cache" / "kagglehub" / "datasets" / "alaakhaled" / "conll003-englishversion" / "versions" / "1"

if default_cache.exists():
    path = default_cache
else:
    import kagglehub
    path = Path(kagglehub.dataset_download("alaakhaled/conll003-englishversion"))

print("Path to dataset files:", path)


In [ ]:
# Keep original CoNLL line structure for parsing (do not flatten into one line)
with open(path / "train.txt", encoding="utf-8") as f:
    data_train = f.read()

with open(path / "valid.txt", encoding="utf-8") as f:
    data_valid = f.read()

with open(path / "test.txt", encoding="utf-8") as f:
    data_test = f.read()

print("Train chars:", len(data_train))
print("Valid chars:", len(data_valid))
print("Test chars:", len(data_test))


In [ ]:
data_train[0:500]

In [ ]:
import random
import spacy
from spacy.training import Example
from spacy.util import minibatch, fix_random_seed
from spacy.scorer import Scorer

fix_random_seed(42)
random.seed(42)

N_EPOCHS = 6
MAX_TRAIN_SENTENCES = 3000
MAX_VALID_SENTENCES = 600
MAX_TEST_SENTENCES = 600

print("spaCy version:", spacy.__version__)


In [ ]:
# Parse CoNLL text (token pos chunk ner) into (tokens, ner_tags) sentences

def parse_conll_text(raw_text):
    sentences = []
    tokens, ner_tags = [], []

    for line in raw_text.splitlines():
        line = line.strip()
        if not line:
            if tokens:
                sentences.append((tokens, ner_tags))
                tokens, ner_tags = [], []
            continue

        if line.startswith("-DOCSTART-"):
            continue

        parts = line.split()
        if len(parts) < 2:
            continue

        token = parts[0]
        ner_tag = parts[-1]
        tokens.append(token)
        ner_tags.append(ner_tag)

    if tokens:
        sentences.append((tokens, ner_tags))

    return sentences

train_sents = parse_conll_text(data_train)[:MAX_TRAIN_SENTENCES]
valid_sents = parse_conll_text(data_valid)[:MAX_VALID_SENTENCES]
test_sents = parse_conll_text(data_test)[:MAX_TEST_SENTENCES]

print("Train sentences:", len(train_sents))
print("Valid sentences:", len(valid_sents))
print("Test sentences:", len(test_sents))


In [ ]:
train_sents[0]

In [ ]:
# Convert CoNLL BIO tags into spaCy Example objects

def bio_to_spans(tags):
    spans = []
    start, label = None, None

    for i, tag in enumerate(tags):
        if tag == "O":
            if start is not None:
                spans.append((start, i - 1, label))
                start, label = None, None
            continue

        prefix, ent = tag.split("-", 1)

        if prefix == "B" or (prefix == "I" and (start is None or ent != label)):
            if start is not None:
                spans.append((start, i - 1, label))
            start, label = i, ent

    if start is not None:
        spans.append((start, len(tags) - 1, label))

    return spans


def sent_to_text_offsets(tokens):
    text = ""
    offsets = []

    for tok in tokens:
        if text:
            text += " "
        s = len(text)
        text += tok
        e = len(text)
        offsets.append((s, e))

    return text, offsets


def build_examples(nlp, sentences):
    examples = []

    for tokens, tags in sentences:
        text, offsets = sent_to_text_offsets(tokens)
        ents = []

        for s_tok, e_tok, label in bio_to_spans(tags):
            s_char = offsets[s_tok][0]
            e_char = offsets[e_tok][1]
            ents.append((s_char, e_char, label))

        doc = nlp.make_doc(text)
        examples.append(Example.from_dict(doc, {"entities": ents}))

    return examples

In [ ]:
# Create a simple spaCy NER model
nlp = spacy.blank("en")
ner = nlp.add_pipe("ner")

for tokens, tags in train_sents:
    for tag in tags:
        if tag != "O":
            ner.add_label(tag.split("-", 1)[1])

train_examples = build_examples(nlp, train_sents)
valid_examples = build_examples(nlp, valid_sents)
test_examples = build_examples(nlp, test_sents)

print("Labels:", ner.labels)
print("Train examples:", len(train_examples))
print("Valid examples:", len(valid_examples))
print("Test examples:", len(test_examples))


In [ ]:
train_examples[4]

In [ ]:
# Train
optimizer = nlp.initialize(get_examples=lambda: train_examples)

for epoch in range(1, N_EPOCHS + 1):
    random.shuffle(train_examples)
    losses = {}

    for batch in minibatch(train_examples, size=8):
        nlp.update(batch, sgd=optimizer, losses=losses, drop=0.25)

    print(f"Epoch {epoch} | NER loss: {losses.get('ner', 0.0):.3f}")

In [ ]:
# Validate/Test with Precision / Recall / F1
def evaluate_split(examples, split_name):
    pred_examples = [Example(nlp(ex.reference.text), ex.reference) for ex in examples]
    scores = Scorer().score(pred_examples)

    print(f"{split_name} Metrics")
    print(f"Precision: {scores['ents_p']:.4f}")
    print(f"Recall:    {scores['ents_r']:.4f}")
    print(f"F1:        {scores['ents_f']:.4f}\n")

evaluate_split(valid_examples, "Validation")
evaluate_split(test_examples, "Test")


In [ ]:
# Save model
nlp.to_disk("simple_conll_ner_model_v2")
print("Saved to: simple_conll_ner_model_v2")


In [ ]:
# Quick prediction example
sample_text = "Christopher Nolan directed Oppenheimer in the United States."
doc = nlp(sample_text)

print(sample_text)
for ent in doc.ents:
    print(ent.text, "->", ent.label_)